In [91]:
import re
import polars as pl

In [92]:
ctd_exposure_events = catalog.load("bronze.ctd.ctd_exposure_events")
mondo_xrefs = catalog.load("bronze.ontology.mondo_xrefs")
mondo_terms = catalog.load("bronze.ontology.mondo_terms")
df_exp_disease = ctd_exposure_events.select(
    ["exposure_stressor_name", "exposure_stressor_id", "disease_name", "disease_id"]
)

[06/11/25 22:12:47] INFO     Loading data from bronze.ctd.ctd_exposure_events             ]8;id=3685;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=796191;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

                    INFO     Loading data from bronze.ontology.mondo_xrefs                ]8;id=614432;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=571795;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

                    INFO     Loading data from bronze.ontology.mondo_terms                ]8;id=929350;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=765278;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

In [93]:
mondo_xrefs.head, mondo_terms.head


(
    <bound method DataFrame.head of shape: (134_206, 2)
┌───────────────┬────────────────────────────┐
│ id            ┆ xref_id                    │
│ ---           ┆ ---                        │
│ str           ┆ str                        │
╞═══════════════╪════════════════════════════╡
│ MONDO_0012295 ┆ MEDGEN:91003               │
│ MONDO_0043339 ┆ SCTID:5724005              │
│ MONDO_0014460 ┆ OMIM:616029                │
│ MONDO_0001129 ┆ MEDGEN:233749              │
│ MONDO_0008943 ┆ MEDGEN:349134              │
│ …             ┆ …                          │
│ MONDO_0007737 ┆ icd11.foundation:518723993 │
│ MONDO_0014948 ┆ MEDGEN:934653              │
│ MONDO_0004232 ┆ MEDGEN:274269              │
│ MONDO_0019034 ┆ UMLS:C0266268              │
│ MONDO_0002486 ┆ DOID:3010                  │
└───────────────┴────────────────────────────┘>,
    <bound method DataFrame.head of shape: (30_204, 3)
┌───────────────┬─────────────────────────────────┬───────┐
│ id            ┆ name   

In [94]:
mesh_xrefs = mondo_xrefs.filter(pl.col("xref_id").str.contains("MESH"))
mesh_xrefs

id,xref_id
str,str
"""MONDO_0012305""","""MESH:C563695"""
"""MONDO_0010019""","""MESH:C562869"""
"""MONDO_0019082""","""MESH:D010391"""
"""MONDO_0005564""","""MESH:D009373"""
"""MONDO_0007914""","""MESH:C563613"""
…,…
"""MONDO_0011563""","""MESH:C565323"""
"""MONDO_0012944""","""MESH:C567245"""
"""MONDO_0007962""","""MESH:C562546"""


In [95]:
df_exp_disease = df_exp_disease.filter(pl.col("disease_id").is_not_null()).with_columns(
    pl.col("disease_id").map_elements(lambda x: f"MESH:{x}").alias("disease_id")
)
df_exp_disease

                    WARNING  /tmp/ipykernel_1136467/2178173091.py:1: MapWithoutReturnDtypeWarning:  ]8;id=835611;file:///usr/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=825717;file:///usr/lib/python3.12/warnings.py#110\110]8;;\
                             Calling `map_elements` without specifying `return_dtype` can lead to                  
                             unpredictable results. Specify `return_dtype` to silence this warning.                
                               df_exp_disease =                                                                    
                             df_exp_disease.filter(pl.col("disease_id").is_not_null()).with_columns                
                             (                                                                                     
                                                                                                                   

exposure_stressor_name,exposure_stressor_id,disease_name,disease_id
str,str,str,str
"""1,1,1-trichloroethane""","""MESH:C024566""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Neoplasms""","""MESH:D009369"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690"""
"""1,2,3,4,7,8-hexachlorodibenzof…","""MESH:C051412""","""Metabolic Syndrome""","""MESH:D024821"""
"""1,2,3-trichloropropane""","""MESH:C009536""","""Leukoencephalopathies""","""MESH:D056784"""
…,…,…,…
"""Zinc""","""MESH:D015032""","""Respiratory Distress Syndrome""","""MESH:D012128"""
"""Ziram""","""MESH:D015039""","""Respiratory Sounds""","""MESH:D012135"""
"""Ziram""","""MESH:D015039""","""Death""","""MESH:D003643"""


In [96]:
df_exp_disease = df_exp_disease.join(
    mesh_xrefs, left_on="disease_id", right_on="xref_id", how="left"
)
df_exp_disease

exposure_stressor_name,exposure_stressor_id,disease_name,disease_id,id
str,str,str,str,str
"""1,1,1-trichloroethane""","""MESH:C024566""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690""","""MONDO_0004976"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Neoplasms""","""MESH:D009369""","""MONDO_0005070"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690""","""MONDO_0004976"""
"""1,2,3,4,7,8-hexachlorodibenzof…","""MESH:C051412""","""Metabolic Syndrome""","""MESH:D024821""","""MONDO_0011565"""
"""1,2,3-trichloropropane""","""MESH:C009536""","""Leukoencephalopathies""","""MESH:D056784""",null
…,…,…,…,…
"""Zinc""","""MESH:D015032""","""Respiratory Distress Syndrome""","""MESH:D012128""","""MONDO_0100130"""
"""Ziram""","""MESH:D015039""","""Respiratory Sounds""","""MESH:D012135""",null
"""Ziram""","""MESH:D015039""","""Death""","""MESH:D003643""",null


In [97]:
df_exp_disease = df_exp_disease.join(
    mondo_terms, left_on="id", right_on="id", how="left"
)
df_exp_disease

exposure_stressor_name,exposure_stressor_id,disease_name,disease_id,id,name,type
str,str,str,str,str,str,str
"""1,1,1-trichloroethane""","""MESH:C024566""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690""","""MONDO_0004976""","""amyotrophic lateral sclerosis""","""CLASS"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Neoplasms""","""MESH:D009369""","""MONDO_0005070""","""neoplasm""","""CLASS"""
"""1,1,2,2-tetrachloroethane""","""MESH:C015530""","""Amyotrophic Lateral Sclerosis""","""MESH:D000690""","""MONDO_0004976""","""amyotrophic lateral sclerosis""","""CLASS"""
"""1,2,3,4,7,8-hexachlorodibenzof…","""MESH:C051412""","""Metabolic Syndrome""","""MESH:D024821""","""MONDO_0011565""","""metabolic syndrome X""","""CLASS"""
"""1,2,3-trichloropropane""","""MESH:C009536""","""Leukoencephalopathies""","""MESH:D056784""",null,null,null
…,…,…,…,…,…,…
"""Zinc""","""MESH:D015032""","""Respiratory Distress Syndrome""","""MESH:D012128""","""MONDO_0100130""","""adult acute respiratory distre…","""CLASS"""
"""Ziram""","""MESH:D015039""","""Respiratory Sounds""","""MESH:D012135""",null,null,null
"""Ziram""","""MESH:D015039""","""Death""","""MESH:D003643""",null,null,null


In [98]:
df_exp_disease = (
    df_exp_disease.rename(
        {
            "exposure_stressor_id": "x_id",
            "exposure_stressor_name": "x_name",
            "disease_id": "y_id",
            "disease_name": "y_name",
        }
    )
    .with_columns(
        [
            pl.lit("exposure").alias("x_type"),
            pl.lit("CTD").alias("x_source"),
            pl.lit("disease").alias("y_type"),
            pl.lit("MONDO").alias("y_source"),
            pl.lit("exposure_disease").alias("relation"),
            pl.lit("linked to").alias("relation_type"),
        ]
    )
    .select(
        [
            "relation",
            "relation_type",
            "x_id",
            "x_type",
            "x_name",
            "x_source",
            "y_id",
            "y_type",
            "y_name",
            "y_source",
        ]
    )
)
df_exp_disease

relation,relation_type,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
str,str,str,str,str,str,str,str,str,str
"""exposure_disease""","""linked to""","""MESH:C024566""","""exposure""","""1,1,1-trichloroethane""","""CTD""","""MESH:D000690""","""disease""","""Amyotrophic Lateral Sclerosis""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:C015530""","""exposure""","""1,1,2,2-tetrachloroethane""","""CTD""","""MESH:D009369""","""disease""","""Neoplasms""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:C015530""","""exposure""","""1,1,2,2-tetrachloroethane""","""CTD""","""MESH:D000690""","""disease""","""Amyotrophic Lateral Sclerosis""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:C051412""","""exposure""","""1,2,3,4,7,8-hexachlorodibenzof…","""CTD""","""MESH:D024821""","""disease""","""Metabolic Syndrome""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:C009536""","""exposure""","""1,2,3-trichloropropane""","""CTD""","""MESH:D056784""","""disease""","""Leukoencephalopathies""","""MONDO"""
…,…,…,…,…,…,…,…,…,…
"""exposure_disease""","""linked to""","""MESH:D015032""","""exposure""","""Zinc""","""CTD""","""MESH:D012128""","""disease""","""Respiratory Distress Syndrome""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:D015039""","""exposure""","""Ziram""","""CTD""","""MESH:D012135""","""disease""","""Respiratory Sounds""","""MONDO"""
"""exposure_disease""","""linked to""","""MESH:D015039""","""exposure""","""Ziram""","""CTD""","""MESH:D003643""","""disease""","""Death""","""MONDO"""
